# Lab 13: YOLO Training

**Module 13** | Read `notes/13-yolo-map.pdf` before running this notebook.

Fine-tune YOLOv8n on COCO128 and read off mAP@0.5 from the validation metrics.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## YOLOv8 training on COCO128

YOLO (You Only Look Once) predicts bounding boxes in a single forward pass. Ultralytics YOLOv8 wraps training, validation, and checkpointing in a high-level API.

COCO128 is a tiny 128-image subset of COCO bundled for smoke tests. On first run Ultralytics downloads both the `yolov8n.pt` weights and the COCO128 images automatically, expect a short delay and network use.


### Train YOLOv8n for 10 epochs

We use the nano variant for speed. Training writes run artifacts under `runs/detect/` by default.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
train_results = model.train(data="coco128.yaml", epochs=10, imgsz=640)

print("Training complete.")
print(f"Best weights: {train_results.save_dir}")


### Validate and read mAP@0.5

`model.val()` re-runs the validation split and returns box metrics. mAP@0.5 (IoU threshold 0.5) is the standard single-threshold detection score reported in many papers and benchmarks.


In [ ]:
metrics = model.val()

map50 = metrics.box.map50
map50_95 = metrics.box.map
print(f"mAP@0.5:   {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")


### Quick inference sanity check

Run the fine-tuned weights on a sample image to confirm the pipeline end-to-end.


In [ ]:
from pathlib import Path

ROOT = Path("..").resolve()
sample_dir = ROOT / "datasets" / "sample_images"
sample = next(iter(sorted(sample_dir.glob("*.jpg"))), None)
if sample:
    preds = model.predict(source=str(sample), imgsz=640, conf=0.25, verbose=False)
    preds[0].show()
    print(f"Ran prediction on {sample.name}")
else:
    print("No sample JPG found, training and validation still completed.")
